# Phase 1 — Feature Analysis Visualization & Discussion Guide

**BMED 712 Track A | Feature Descriptive Statistics**

This notebook generates publication-ready figures from Fatemah's Phase 1 descriptive statistics output. The goal is to distill **216 features** into a clear, presentable narrative.

**Strategy:**
1. **Rank features** by effect size (η² from Kruskal-Wallis) — focus on top 15–20
2. **Group by sensor** to answer: which sensor placement matters most?
3. **Group by signal type** (Acc / FreeAcc / Gyr) to answer: which signal is most discriminative?
4. **Show pairwise separation** (Cohen's d) to answer: which features best separate neuro vs ortho?
5. **Correlation filtering** to identify redundant features

**Input:** CSV files from `PHASE1_Feature_Analysis_COMPLETE/` subdirectories  
**Output:** Figures + summary tables ready for the report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Configuration ──
BASE_DIR = Path(".")  # this notebook lives inside PHASE1_Feature_Analysis_COMPLETE
FOCUS_CONFIG = "Full_Gait_6s_ov50"  # primary config to analyze (change as needed)
FIG_DIR = BASE_DIR / "report_figures"
FIG_DIR.mkdir(exist_ok=True)

# Color scheme
SENSOR_COLORS = {"HE": "#3498db", "LB": "#2ecc71", "LF": "#e67e22", "RF": "#e74c3c"}
CHANNEL_COLORS = {"Acc": "#9b59b6", "FreeAcc": "#1abc9c", "Gyr": "#e74c3c"}
GROUP_COLORS = {"healthy": "#2ecc71", "neuro": "#e74c3c", "ortho": "#3498db"}

print(f"Focus config: {FOCUS_CONFIG}")
print(f"Figures will be saved to: {FIG_DIR}")

## 1. Load Data — Kruskal-Wallis, Effect Sizes, Descriptive Stats

In [ ]:
# Load the three key CSVs for our focus config
kw = pd.read_csv(BASE_DIR / FOCUS_CONFIG / f"kruskal_wallis_{FOCUS_CONFIG}.csv")
es = pd.read_csv(BASE_DIR / FOCUS_CONFIG / f"effect_sizes_{FOCUS_CONFIG}.csv")
ds = pd.read_csv(BASE_DIR / FOCUS_CONFIG / f"descriptive_stats_{FOCUS_CONFIG}.csv")

# Parse sensor, channel, axis, metric from feature name
def parse_feature(name):
    parts = name.split("_")
    # Format: SENSOR_CHANNEL_AXIS_METRIC  e.g. HE_Acc_X_mean
    if len(parts) >= 4:
        return parts[0], parts[1], parts[2], "_".join(parts[3:])
    return parts[0], "", "", name

kw[["Sensor", "Channel", "Axis", "Metric"]] = pd.DataFrame(
    kw["Feature"].apply(parse_feature).tolist(), index=kw.index
)

print(f"Kruskal-Wallis: {len(kw)} features")
print(f"  Significant (p<0.05): {(kw['p_value'] < 0.05).sum()}")
print(f"  Large effect (η²>0.06): {(kw['eta_squared'] > 0.06).sum()}")
print(f"\nEffect sizes: {len(es)} features, columns: {list(es.columns)}")
print(f"\nDescriptive stats: {len(ds)} features × {len(ds.columns)} columns")
print(f"\nTop 5 features by η²:")
display(kw.nlargest(5, "eta_squared")[["Feature", "H_Statistic", "p_value", "eta_squared", "Effect_Size"]])

## 2. Figure 1 — Top 20 Features by Effect Size (η²)

This is the **most important figure** for the report. It ranks features by Kruskal-Wallis η² and colors them by sensor placement, instantly showing which sensors and features carry the most discriminative power.

In [ ]:
top20 = kw.nlargest(20, "eta_squared").copy()
top20 = top20.iloc[::-1]  # reverse for horizontal bar (top at top)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [SENSOR_COLORS.get(s, "gray") for s in top20["Sensor"]]
bars = ax.barh(range(len(top20)), top20["eta_squared"], color=colors, edgecolor="white", linewidth=0.5)

# Annotate η² values
for i, (val, feat) in enumerate(zip(top20["eta_squared"], top20["Feature"])):
    ax.text(val + 0.003, i, f"{val:.3f}", va="center", fontsize=8)

ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["Feature"], fontsize=9)
ax.set_xlabel("η² (Kruskal-Wallis Effect Size)", fontsize=11)
ax.set_title(f"Top 20 Features by Effect Size — {FOCUS_CONFIG}", fontsize=13, fontweight="bold")

# Legend
handles = [mpatches.Patch(color=c, label=s) for s, c in SENSOR_COLORS.items()]
ax.legend(handles=handles, title="Sensor", loc="lower right", fontsize=9)

# Effect size thresholds
ax.axvline(0.01, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
ax.axvline(0.06, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axvline(0.14, color="gray", linestyle="-", linewidth=0.8, alpha=0.5)
ax.text(0.012, -0.8, "small", fontsize=7, color="gray")
ax.text(0.062, -0.8, "medium", fontsize=7, color="gray")
ax.text(0.142, -0.8, "large", fontsize=7, color="gray")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_top20_features_eta_squared.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig1_top20_features_eta_squared.png")

## 3. Figure 2 — Sensor-Level Summary (Average η² per Sensor)

This collapses 216 features into a **per-sensor overview**, answering: *"Which sensor placement is most informative for gait classification?"*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel A: Average η² per sensor ---
ax = axes[0]
sensor_avg = kw.groupby("Sensor")["eta_squared"].agg(["mean", "std", "count"]).reindex(["HE", "LB", "LF", "RF"])
bars = ax.bar(sensor_avg.index, sensor_avg["mean"],
              yerr=sensor_avg["std"], capsize=4,
              color=[SENSOR_COLORS[s] for s in sensor_avg.index],
              edgecolor="white", linewidth=0.5)
ax.set_ylabel("Mean η²", fontsize=11)
ax.set_title("A) Average Effect Size by Sensor", fontsize=12, fontweight="bold")
ax.axhline(0.06, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(3.3, 0.062, "medium\nthreshold", fontsize=7, color="gray", va="bottom")
for i, (s, row) in enumerate(sensor_avg.iterrows()):
    ax.text(i, row["mean"] + row["std"] + 0.003, f'{row["mean"]:.3f}', ha="center", fontsize=9)

# --- Panel B: Number of large-effect features per sensor ---
ax = axes[1]
large = kw[kw["eta_squared"] > 0.06].groupby("Sensor").size().reindex(["HE", "LB", "LF", "RF"], fill_value=0)
total = kw.groupby("Sensor").size().reindex(["HE", "LB", "LF", "RF"], fill_value=0)
ax.bar(large.index, large.values,
       color=[SENSOR_COLORS[s] for s in large.index],
       edgecolor="white", linewidth=0.5)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("B) Features with Large Effect (η² > 0.06)", fontsize=12, fontweight="bold")
for i, (n_large, n_total) in enumerate(zip(large.values, total.values)):
    ax.text(i, n_large + 0.3, f"{n_large}/{n_total}", ha="center", fontsize=10)

# --- Panel C: Average η² per channel ---
ax = axes[2]
chan_avg = kw.groupby("Channel")["eta_squared"].agg(["mean", "std"]).reindex(["Acc", "FreeAcc", "Gyr"])
bars = ax.bar(chan_avg.index, chan_avg["mean"],
              yerr=chan_avg["std"], capsize=4,
              color=[CHANNEL_COLORS[c] for c in chan_avg.index],
              edgecolor="white", linewidth=0.5)
ax.set_ylabel("Mean η²", fontsize=11)
ax.set_title("C) Average Effect Size by Signal Type", fontsize=12, fontweight="bold")
ax.axhline(0.06, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
for i, (c, row) in enumerate(chan_avg.iterrows()):
    ax.text(i, row["mean"] + row["std"] + 0.003, f'{row["mean"]:.3f}', ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_sensor_channel_summary.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig2_sensor_channel_summary.png")

## 4. Figure 3 — Sensor × Channel Heatmap

A **heatmap of average η²** for every sensor–channel combination, showing which sensor-signal pairs are most discriminative. Each cell aggregates across 3 axes × 6 metrics = 18 features.

In [ ]:
pivot = kw.pivot_table(index="Sensor", columns="Channel", values="eta_squared", aggfunc="mean")
pivot = pivot.reindex(index=["HE", "LB", "LF", "RF"], columns=["Acc", "FreeAcc", "Gyr"])

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto", vmin=0)

# Annotate cells
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        color = "white" if val > pivot.values.max() * 0.6 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=12, fontweight="bold", color=color)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=11)
ax.set_xlabel("Signal Type", fontsize=11)
ax.set_ylabel("Sensor", fontsize=11)
ax.set_title(f"Mean η² by Sensor × Signal Type — {FOCUS_CONFIG}", fontsize=13, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean η²", fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_sensor_channel_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig3_sensor_channel_heatmap.png")

## 5. Figure 4 — Pairwise Cohen's d (Top 15 Features)

Shows **which features best separate each pair of groups** (healthy vs neuro, healthy vs ortho, neuro vs ortho). This answers the clinical question: *"Are the same features that detect neurological gait also useful for orthopedic gait?"*

In [ ]:
# Merge effect sizes with KW to get top features
es_kw = kw.merge(es, on="Feature", how="left")

# Get top 15 by η²
top15 = es_kw.nlargest(15, "eta_squared").copy()

fig, ax = plt.subplots(figsize=(12, 7))

y_pos = np.arange(len(top15))
width = 0.25

pairs = [
    ("d_healthy_vs_neuro", "Healthy vs Neuro", "#e74c3c"),
    ("d_healthy_vs_ortho", "Healthy vs Ortho", "#3498db"),
    ("d_neuro_vs_ortho",   "Neuro vs Ortho",   "#9b59b6"),
]

for i, (col, label, color) in enumerate(pairs):
    vals = top15[col].values
    ax.barh(y_pos + i * width, vals, width, label=label, color=color, alpha=0.85, edgecolor="white")

ax.set_yticks(y_pos + width)
ax.set_yticklabels(top15["Feature"].values[::-1] if False else top15["Feature"].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Cohen's d", fontsize=11)
ax.set_title(f"Pairwise Cohen's d — Top 15 Features by η² ({FOCUS_CONFIG})", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="lower right")
ax.axvline(0, color="black", linewidth=0.5)

# Effect size guidelines
for threshold, label in [(0.2, "small"), (0.5, "medium"), (0.8, "large")]:
    ax.axvline(threshold, color="gray", linestyle=":", linewidth=0.7, alpha=0.5)
    ax.axvline(-threshold, color="gray", linestyle=":", linewidth=0.7, alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_pairwise_cohens_d_top15.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig4_pairwise_cohens_d_top15.png")

## 6. Figure 5 — Radar Chart: Sensor Discriminative Profile

A radar (spider) chart showing each sensor's average η² across the 6 feature metrics (mean, std, rms, dom_freq, spec_centroid, spec_power). This reveals whether certain sensors excel at specific feature types.

In [ ]:
metrics = ["mean", "std", "rms", "dom_freq", "spec_centroid", "spec_power"]
radar_data = kw[kw["Metric"].isin(metrics)].pivot_table(
    index="Sensor", columns="Metric", values="eta_squared", aggfunc="mean"
).reindex(index=["HE", "LB", "LF", "RF"], columns=metrics)

# Radar plot
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for sensor in ["HE", "LB", "LF", "RF"]:
    values = radar_data.loc[sensor].values.tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=sensor, color=SENSOR_COLORS[sensor])
    ax.fill(angles, values, alpha=0.1, color=SENSOR_COLORS[sensor])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
ax.set_title(f"Sensor Discriminative Profile — {FOCUS_CONFIG}", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_sensor_radar_chart.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig5_sensor_radar_chart.png")

## 7. Figure 6 — Top 5 Features: Group Comparison (Descriptive Stats)

Using the descriptive statistics to show **per-group mean ± std** for the top 5 most discriminative features. This is the "zoomed-in" view that complements the boxplots Fatemah already generated.

In [ ]:
top5_names = kw.nlargest(5, "eta_squared")["Feature"].tolist()
top5_ds = ds[ds["Feature"].isin(top5_names)].copy()
# Preserve ranking order
top5_ds["rank"] = top5_ds["Feature"].map({f: i for i, f in enumerate(top5_names)})
top5_ds = top5_ds.sort_values("rank")

fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=False)

for idx, (_, row) in enumerate(top5_ds.iterrows()):
    ax = axes[idx]
    groups = ["healthy", "neuro", "ortho"]
    means = [row[f"{g}_Mean"] for g in groups]
    stds = [row[f"{g}_Std"] for g in groups]
    colors = [GROUP_COLORS[g] for g in groups]

    bars = ax.bar(groups, means, yerr=stds, capsize=5,
                  color=colors, edgecolor="white", linewidth=0.5, alpha=0.85)
    ax.set_title(row["Feature"], fontsize=9, fontweight="bold")
    ax.tick_params(axis="x", labelsize=8)

    # Add η² annotation
    eta = kw[kw["Feature"] == row["Feature"]]["eta_squared"].values[0]
    ax.text(0.95, 0.95, f"η²={eta:.3f}", transform=ax.transAxes,
            fontsize=8, ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

axes[0].set_ylabel("Feature Value (mean ± std)", fontsize=10)
fig.suptitle(f"Top 5 Features — Group Comparison ({FOCUS_CONFIG})", fontsize=14, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig6_top5_group_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig6_top5_group_comparison.png")

## 8. Figure 7 — Cross-Config Comparison (All 12 Configurations)

Compares the number of significant features and top η² across all 12 phase × window × overlap configurations. This shows whether the statistical findings are robust or config-dependent.

In [ ]:
# Load summary across all configs
summary = pd.read_csv(BASE_DIR / "SUMMARY_Phase1_Analysis_COMPLETE.csv")
print("Summary of all 12 configurations:")
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: n_significant per config
ax = axes[0]
summary_sorted = summary.sort_values("n_significant", ascending=True)
colors = summary_sorted["config"].apply(
    lambda c: "#e74c3c" if "UTurn" in c else "#3498db" if "Post" in c 
    else "#2ecc71" if "Pre" in c else "#f39c12"
).values
ax.barh(range(len(summary_sorted)), summary_sorted["n_significant"],
        color=colors, edgecolor="white")
ax.set_yticks(range(len(summary_sorted)))
ax.set_yticklabels(summary_sorted["config"], fontsize=8)
ax.set_xlabel("Number of Significant Features (p < 0.05)")
ax.set_title("A) Significant Features per Configuration", fontsize=12, fontweight="bold")
ax.axvline(216, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.text(216, -0.5, "max=216", fontsize=7, color="gray")

# Panel B: top η² per config
ax = axes[1]
summary_sorted2 = summary.sort_values("top_eta_squared", ascending=True)
colors2 = summary_sorted2["config"].apply(
    lambda c: "#e74c3c" if "UTurn" in c else "#3498db" if "Post" in c 
    else "#2ecc71" if "Pre" in c else "#f39c12"
).values
ax.barh(range(len(summary_sorted2)), summary_sorted2["top_eta_squared"],
        color=colors2, edgecolor="white")
ax.set_yticks(range(len(summary_sorted2)))
ax.set_yticklabels(summary_sorted2["config"], fontsize=8)
ax.set_xlabel("Top Feature η²")
ax.set_title("B) Strongest Feature Effect Size per Configuration", fontsize=12, fontweight="bold")

# Annotate top feature name
for i, (_, row) in enumerate(summary_sorted2.iterrows()):
    ax.text(row["top_eta_squared"] + 0.005, i, row["top_feature"], fontsize=6, va="center")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#f39c12", label="Full Gait"),
    Patch(facecolor="#2ecc71", label="Pre U-Turn"),
    Patch(facecolor="#3498db", label="Post U-Turn"),
    Patch(facecolor="#e74c3c", label="U-Turn"),
]
axes[0].legend(handles=legend_elements, fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig7_cross_config_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig7_cross_config_comparison.png")

## 9. Summary Table — Compact Sensor Report for the Paper

This generates a **publication-ready summary table** that compresses 216 features into one table. Perfect for the Methods or Results section.

In [ ]:
# Build compact summary table per sensor
rows = []
for sensor in ["HE", "LB", "LF", "RF"]:
    sub = kw[kw["Sensor"] == sensor]
    n_sig = (sub["p_value"] < 0.05).sum()
    n_large = (sub["eta_squared"] > 0.06).sum()
    n_total = len(sub)
    mean_eta = sub["eta_squared"].mean()
    top_feat = sub.loc[sub["eta_squared"].idxmax()]

    rows.append({
        "Sensor": sensor,
        "Total Features": n_total,
        "Significant (p<0.05)": f"{n_sig}/{n_total}",
        "Large Effect (η²>0.06)": n_large,
        "Mean η²": f"{mean_eta:.3f}",
        "Top Feature": top_feat["Feature"],
        "Top η²": f"{top_feat['eta_squared']:.3f}",
    })

table = pd.DataFrame(rows)
print("=== Compact Sensor Summary Table ===")
display(table)

# Also per channel
rows_ch = []
for ch in ["Acc", "FreeAcc", "Gyr"]:
    sub = kw[kw["Channel"] == ch]
    n_sig = (sub["p_value"] < 0.05).sum()
    n_large = (sub["eta_squared"] > 0.06).sum()
    mean_eta = sub["eta_squared"].mean()
    top_feat = sub.loc[sub["eta_squared"].idxmax()]
    rows_ch.append({
        "Channel": ch,
        "Significant": f"{n_sig}/{len(sub)}",
        "Large Effect": n_large,
        "Mean η²": f"{mean_eta:.3f}",
        "Top Feature": top_feat["Feature"],
        "Top η²": f"{top_feat['eta_squared']:.3f}",
    })

table_ch = pd.DataFrame(rows_ch)
print("\n=== Compact Channel Summary Table ===")
display(table_ch)

# Save to CSV for report
table.to_csv(FIG_DIR / "summary_table_sensors.csv", index=False)
table_ch.to_csv(FIG_DIR / "summary_table_channels.csv", index=False)
print("\n✓ Saved: summary_table_sensors.csv, summary_table_channels.csv")

## 10. Figure 8 — Feature Metric Breakdown (Which Stat Type Is Most Discriminative?)

Bar chart showing mean η² for each of the 6 feature metrics (mean, std, rms, dom_freq, spec_centroid, spec_power), broken down by sensor. Answers: *"Is frequency-domain or time-domain information more useful?"*

In [ ]:
metrics = ["mean", "std", "rms", "dom_freq", "spec_centroid", "spec_power"]
metric_labels = ["Mean", "Std", "RMS", "Dom. Freq.", "Spec. Centroid", "Spec. Power"]

pivot_metric = kw[kw["Metric"].isin(metrics)].pivot_table(
    index="Metric", columns="Sensor", values="eta_squared", aggfunc="mean"
).reindex(index=metrics, columns=["HE", "LB", "LF", "RF"])

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(metrics))
width = 0.2

for i, sensor in enumerate(["HE", "LB", "LF", "RF"]):
    vals = pivot_metric[sensor].values
    ax.bar(x + i * width, vals, width, label=sensor,
           color=SENSOR_COLORS[sensor], edgecolor="white", alpha=0.85)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_ylabel("Mean η²", fontsize=11)
ax.set_title(f"Feature Metric Discriminability by Sensor — {FOCUS_CONFIG}", fontsize=13, fontweight="bold")
ax.legend(title="Sensor", fontsize=10)
ax.axhline(0.06, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.text(5.6, 0.062, "medium\nthreshold", fontsize=7, color="gray")

# Highlight time vs frequency domain
ax.axvspan(-0.3, 2.8, alpha=0.05, color="blue")
ax.axvspan(2.8, 5.8, alpha=0.05, color="red")
ax.text(1.0, ax.get_ylim()[1] * 0.95, "Time-domain", fontsize=9, color="blue", alpha=0.6, ha="center")
ax.text(4.2, ax.get_ylim()[1] * 0.95, "Frequency-domain", fontsize=9, color="red", alpha=0.6, ha="center")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig8_metric_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: fig8_metric_breakdown.png")

## 11. Discussion Guide — How to Present 216 Features

### Narrative Structure for the Report

When writing the results section, organize around **clinical questions**, not raw numbers:

1. **"How many features are informative?"**
   - Report: "X/216 features showed significant group differences (Kruskal-Wallis, p<0.05). Of these, Y features exhibited medium-to-large effect sizes (η² > 0.06)."
   - Use the summary tables above.

2. **"Which sensor placement is most informative?"**
   - Lead with the sensor summary (Fig 2A): HE (head) has the highest mean η².
   - This aligns with our ML finding that HE alone achieves 74% BAcc.
   - Clinical implication: a single head-mounted IMU may suffice for screening.

3. **"Which signal types carry the most discriminative power?"**
   - Use Fig 3 (heatmap) and Fig 8 (metric breakdown).
   - Compare time-domain (mean/std/rms) vs frequency-domain (dom_freq/spec_centroid/spec_power).

4. **"What separates neurological from orthopedic gait?"**
   - Use Fig 4 (pairwise Cohen's d) to show features with large healthy-vs-neuro d but small healthy-vs-ortho d (and vice versa).
   - These "differential features" are clinically valuable.

5. **"Are these findings robust across windowing configurations?"**
   - Use Fig 7 (cross-config) to show consistency.

### Key Talking Points
- **Don't apologize for 216 features** — it's a strength. Frame it as "comprehensive feature extraction enables systematic analysis."
- **The figures do the heavy lifting** — one sentence per figure is enough in the text.
- **Connect to ML results** — "The statistical finding that HE features have the largest effect sizes is corroborated by our sensor ablation experiment showing HE alone retains 97.8% of classification performance."

In [ ]:
print("=" * 60)
print("NOTEBOOK COMPLETE — Generated Figures Summary")
print("=" * 60)
print()
import os
for f in sorted(FIG_DIR.iterdir()):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f.name:45s}  ({size_kb:.0f} KB)" if f.exists() else f"  {f.name} — NOT YET GENERATED (run cells above)")
print()
print("All figures saved to:", FIG_DIR)
print()
print("Recommended figures for the report:")
print("  1. fig1_top20_features_eta_squared.png   — Main ranking figure")
print("  2. fig2_sensor_channel_summary.png       — Sensor + channel overview")
print("  3. fig3_sensor_channel_heatmap.png       — Compact sensor×channel view")
print("  4. fig4_pairwise_cohens_d_top15.png      — Group separation detail")
print("  5. fig5_sensor_radar_chart.png            — Sensor profile (optional)")
print("  6. fig6_top5_group_comparison.png         — Zoomed-in group means")
print("  7. fig7_cross_config_comparison.png       — Robustness check")
print("  8. fig8_metric_breakdown.png              — Time vs frequency domain")